# 05a -- Confounder Analysis

**Purpose:** Analyse per-category false positive rates on the training set to
identify confounders (categories the model consistently misclassifies as flood).
This informs hard negative mining (Step 05b/05c).

**Compute:** Colab T4 GPU (for inference on training set).

**Estimated runtime:** ~5-10 minutes.

**Script:** `scripts/analyze_confounders.py`

**Prerequisites:** Run `notebooks/02_prepare_confounder_data.ipynb` first to
download and split the four visual confounder categories:

| Category | Visual similarity to flood | Source(s) |
|---|---|---|
| Swimmingpool | High — flat reflective water surface | Places365 + Open Images v7 |
| River | High — flowing water, often muddy | ATLANTIS + RIWA + WaterNet + LuFI-RiverSnap |
| Lake | High — still water surface / shoreline | ATLANTIS + WaterNet |
| Fountain | Medium — water in motion, urban setting | Open Images v7 + ADE20K |

**Note:** Replace `TIMESTAMP` in model paths below with the actual filename
from training (e.g., `efficientnet_baseline_20260315_143000.keras`).

In [ ]:
# Mount Google Drive and verify GPU runtime
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'T4' in gpu_info.stdout:
    print('Runtime: Colab T4 GPU')
elif 'failed' in gpu_info.stderr.lower() or gpu_info.returncode != 0:
    print('Runtime: CPU only -- switch to a GPU runtime!')
else:
    print(f'Runtime: GPU detected\n{gpu_info.stdout[:200]}')

In [ ]:
%cd /content/drive/MyDrive/imagevalidation2

In [ ]:
# Verify that all four confounder categories are present in the training set
from pathlib import Path

train_nf = Path('/content/drive/MyDrive/FloodingDataset2/train/non_flood')
EXPECTED_CATS = ['Swimmingpool', 'River', 'Lake', 'Fountain']

found = {p.name for p in train_nf.iterdir() if p.is_dir()} if train_nf.exists() else set()
# Also check flat directory (files named Category_NNNN.jpg)
if train_nf.exists():
    prefixes = {f.name.split('_')[0] for f in train_nf.glob('*.jpg')}
    found |= prefixes

print('Confounder category check (train/non_flood/):')
for cat in EXPECTED_CATS:
    status = '✓' if any(cat.lower() in x.lower() for x in found) else '✗ MISSING — run notebook 02'
    n = len(list(train_nf.glob(f'{cat}_*.jpg'))) if train_nf.exists() else 0
    print(f'  {cat:<15} {status}  (train images: {n})')

## Run confounder analysis — EfficientNetB0

In [ ]:
!python scripts/analyze_confounders.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

## Run confounder analysis — ResNet50

In [ ]:
!python scripts/analyze_confounders.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

## Display results — both architectures

In [ ]:
import pandas as pd
from IPython.display import display

# Highlight the four water-body confounder categories
WATER_CONFOUNDERS = {'swimmingpool', 'river', 'lake', 'fountain'}

for arch in ['efficientnet', 'resnet50']:
    csv_path = f'./results/tables/confounder_fp_rates_{arch}.csv'
    try:
        df = pd.read_csv(csv_path)
        # Flag water-body confounders
        if 'category' in df.columns:
            df['water_confounder'] = df['category'].str.lower().isin(WATER_CONFOUNDERS)
        print(f'\n=== {arch.upper()} Confounder FP Rates ===')
        display(df.sort_values('fp_rate', ascending=False))
    except FileNotFoundError:
        print(f'Results not found at {csv_path} -- run the analysis cells above first.')